# 🧠 코랩 전이학습으로 나만의 AI 만들기 → 포니봇 주행 제어

**AI 기반 자율로봇 시스템 구현 과정 — 전이학습(Transfer Learning) 실습**

이 노트북에서 하는 일:

1. **데이터 수집** — 방향 표지판(또는 손 모양) 사진을 클래스별로 모읍니다
2. **전이학습** — 구글이 미리 학습해 둔 **MobileNetV2**에 우리 데이터만 얹어 빠르게 학습
3. **변환** — 학습된 모델을 브라우저용 **TensorFlow.js** 형식으로 변환
4. **다운로드** — 변환된 모델을 내려받아 GitHub Pages에 올리면, 웹앱이 포니봇을 제어합니다

```
[코랩: 학습]  →  [TF.js 변환]  →  [GitHub Pages 배포]  →  [웹앱: 카메라 분류]  →  [BLE]  →  [포니봇 주행]
```

> ⏱ 예상 소요: 데이터 수집 15분 + 학습 5분 + 변환/배포 10분

---
## 클래스(폴더 이름) 규칙 — 매우 중요! ⚠️

폴더 이름이 **그대로 로봇 명령**이 됩니다. 아래 5개 이름을 정확히 사용하세요.

| 폴더 이름 | 로봇 동작 | 촬영할 것 (예시) |
|---|---|---|
| `forward` | 전진 | ↑ 화살표 카드 |
| `backward` | 후진 | ↓ 화살표 카드 |
| `left` | 좌회전 | ← 화살표 카드 |
| `right` | 우회전 | → 화살표 카드 |
| `stop` | 정지 | **아무것도 없는 배경** (카드 없이) |

`stop`을 배경 사진으로 하면, 카드를 치우는 순간 로봇이 멈춰서 안전합니다.

---
# 1단계. 준비 — 라이브러리 불러오기

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os, json, shutil, zipfile

print("TensorFlow 버전:", tf.__version__)

# ===== 설정 =====
DATA_DIR = "datasets"           # 사진이 모일 폴더
IMG_SIZE = 224                 # MobileNetV2 입력 크기
CLASSES  = ["backward", "forward", "left", "right", "stop"]  # 알파벳순 (자동 정렬과 동일)

for c in CLASSES:
    os.makedirs(os.path.join(DATA_DIR, c), exist_ok=True)
print("폴더 준비 완료:", os.listdir(DATA_DIR))

---
# 2단계. 데이터 수집

두 가지 방법 중 **하나**를 선택하세요. (섞어서 사용해도 됩니다)

- **방법 A** — 폰/카메라로 미리 찍은 사진을 **ZIP으로 업로드** (권장: 조명·배경 다양하게)
- **방법 B** — 코랩에서 **노트북 웹캠으로 바로 촬영**

> 📸 **좋은 데이터 만들기 팁**
> - 클래스당 **최소 30장, 권장 50장 이상**
> - 카드의 **거리·각도·위치**를 조금씩 바꿔가며 촬영
> - 실제 로봇을 조종할 장소와 **비슷한 배경/조명**에서 촬영하면 인식률이 올라갑니다

## 방법 A — ZIP 업로드

폰에서 찍은 사진을 아래처럼 폴더로 정리해 `dataset.zip`으로 압축한 뒤 업로드하세요.

```
dataset.zip
 └─ dataset/
     ├─ forward/   img001.jpg ...
     ├─ backward/  ...
     ├─ left/      ...
     ├─ right/     ...
     └─ stop/      ...
```

In [ ]:
# 방법 A: dataset.zip 업로드 (방법 B만 쓸 거면 이 셀은 건너뛰세요)
from google.colab import files

uploaded = files.upload()   # 파일 선택 창이 뜹니다 → dataset.zip 선택
for name in uploaded:
    with zipfile.ZipFile(name) as z:
        z.extractall(".")
    print(name, "압축 해제 완료")

In [ ]:
# 데이터 개수 확인 — 클래스당 30장 이상을 목표로!
print("클래스별 이미지 수")
print("-" * 30)
total = 0
for c in CLASSES:
    n = len(os.listdir(os.path.join(DATA_DIR, c)))
    total += n
    bar = "█" * (n // 5)
    print(f"{c:>9s} : {n:4d}장  {bar}")
print("-" * 30)
print(f"{'합계':>8s} : {total:4d}장")

In [ ]:
# 샘플 이미지 미리보기
import random
from PIL import Image

fig, axes = plt.subplots(1, len(CLASSES), figsize=(15, 3))
for ax, c in zip(axes, CLASSES):
    imgs = os.listdir(os.path.join(DATA_DIR, c))
    if imgs:
        img = Image.open(os.path.join(DATA_DIR, c, random.choice(imgs)))
        ax.imshow(img)
    ax.set_title(c)
    ax.axis('off')
plt.show()

---
# 3단계. 데이터셋 만들기 (학습 80% / 검증 20%)

In [ ]:
BATCH = 16
SEED  = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH)

class_names = train_ds.class_names
print("클래스 순서(모델 출력 순서):", class_names)
assert class_names == CLASSES, "폴더 이름을 확인하세요!"

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(200).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

---
# 4단계. 전이학습 모델 만들기

**MobileNetV2**는 이미지 140만 장(ImageNet)으로 미리 학습된 모델입니다.
이 모델의 '눈'(특징 추출부)은 **얼려서(freeze)** 그대로 쓰고,
마지막 분류층만 우리 데이터 5개 클래스로 **새로 학습**합니다.

```
입력(224×224) → [Rescaling] → [MobileNetV2 ❄️동결] → [GlobalAveragePooling] → [Dropout] → [Dense 5 softmax]
```

> 💡 `Rescaling` 층을 모델 안에 넣었기 때문에, 웹앱에서는 카메라 픽셀(0~255)을 **그대로** 넣으면 됩니다.

In [ ]:
# 데이터 증강: 같은 사진도 조금씩 다르게 보여줘서 과적합 방지
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
], name="augment")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,          # 분류층은 떼고 특징 추출부만
    weights="imagenet")         # 미리 학습된 가중치
base_model.trainable = False    # ❄️ 동결 — 이것이 전이학습의 핵심!

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = tf.keras.layers.Rescaling(1./127.5, offset=-1)(x)   # 0~255 → -1~1
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(CLASSES), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()
print(f"\n학습 가능한 파라미터가 전체의 극히 일부인 것을 확인하세요 (전이학습!)")

In [ ]:
# 학습! (클래스당 50장 기준 약 1~2분)
EPOCHS = 12
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

In [ ]:
# 학습 곡선 확인 — val_accuracy가 0.9 이상이면 훌륭!
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy'); plt.legend(); plt.grid(alpha=.3)
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss'); plt.legend(); plt.grid(alpha=.3)
plt.show()

val_acc = history.history['val_accuracy'][-1]
print(f"최종 검증 정확도: {val_acc*100:.1f}%")
if val_acc < 0.85:
    print("⚠️ 정확도가 낮아요 → 사진을 더 모으거나(클래스당 50장+), 배경/조명을 다양하게 해보세요")

## (선택) 미세조정 — 정확도를 더 끌어올리기

동결했던 MobileNetV2의 **끝부분 일부만 녹여서** 아주 작은 학습률로 다시 학습합니다.
검증 정확도가 이미 충분히 높으면 건너뛰어도 됩니다.

In [ ]:
# (선택) 미세조정
base_model.trainable = True
for layer in base_model.layers[:-30]:   # 끝 30개 층만 학습
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),   # 학습률 100배 낮춤!
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=5)
print(f"미세조정 후 검증 정확도: {history_ft.history['val_accuracy'][-1]*100:.1f}%")

---
# 5단계. 검증 이미지로 예측 테스트

In [ ]:
# 검증 데이터 한 배치를 예측해서 눈으로 확인
for images, labels in val_ds.take(1):
    preds = model.predict(images, verbose=0)
    plt.figure(figsize=(14, 7))
    n = min(10, len(images))
    for i in range(n):
        plt.subplot(2, 5, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        p = np.argmax(preds[i]); conf = preds[i][p] * 100
        correct = (p == labels[i].numpy())
        plt.title(f"{class_names[p]} {conf:.0f}%",
                  color=("green" if correct else "red"))
        plt.axis('off')
    plt.show()

---
# 6단계. TensorFlow.js 변환 → 다운로드

브라우저(웹앱)에서 돌아가도록 **TF.js 그래프 모델**로 변환하고,
클래스 이름이 담긴 `labels.json`과 함께 ZIP으로 내려받습니다.

> 🔴 **아래 셀 실행 후 빨간 `ERROR: pip's dependency resolver...` 메시지가 떠도 정상입니다!**
> tensorflowjs가 자기 전용 TensorFlow 버전을 함께 설치하면서, 우리 실습과 무관한
> 코랩 기본 패키지들(tensorflow-text, bigquery 등)이 불평하는 것뿐입니다. **그대로 진행하세요.**
>
> ⚠️ 단, 이 셀 실행 후에는 **앞쪽 학습 셀을 다시 실행하지 마세요.**
> 재학습이 필요하면 메뉴 **런타임 → 세션 다시 시작** 후 처음부터 실행합니다.

In [ ]:
# tensorflowjs 설치 (1~2분)
# 빨간 ERROR(dependency resolver) 메시지는 무시하세요 — 설치는 정상 완료됩니다.
!pip install -q tensorflowjs

# 설치 확인: 버전이 출력되면 성공!
!tensorflowjs_converter --version

In [ ]:
# [수정판] 증강 층을 뺀 '추론 전용 모델'을 만들어 내보내기
# (학습된 가중치는 base_model과 Dense 층에 들어있으므로 그대로 재사용됩니다)

dense = model.layers[-1]                      # 학습된 분류층 (Dense)
assert isinstance(dense, tf.keras.layers.Dense), "마지막 층이 Dense가 아닙니다"

infer_in  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.layers.Rescaling(1./127.5, offset=-1)(infer_in)   # 0~255 → -1~1
x = base_model(x, training=False)             # ❄️ 학습된 특징 추출부 재사용
x = tf.keras.layers.GlobalAveragePooling2D()(x)
infer_out = dense(x)                          # 학습된 분류층 재사용
inference_model = tf.keras.Model(infer_in, infer_out)

# 원본 모델과 예측이 같은지 검증 (증강·드롭아웃은 추론 시 어차피 꺼져 있음)
for images, labels_batch in val_ds.take(1):
    p1 = model.predict(images, verbose=0)
    p2 = inference_model.predict(images, verbose=0)
    print("예측 일치 여부:", np.allclose(p1, p2, atol=1e-5))

inference_model.export("saved_model")

!tensorflowjs_converter \
    --input_format=tf_saved_model \
    --output_format=tfjs_graph_model \
    saved_model tfjs_model

# ★ 변환 성공 검증: model.json이 실제로 생겼는지 확인
assert os.path.exists("tfjs_model/model.json"), \
    "변환 실패! 위 로그의 에러 메시지를 확인하세요."

with open("tfjs_model/labels.json", "w") as f:
    json.dump({"labels": class_names, "img_size": IMG_SIZE}, f)

print("변환 완료! 생성된 파일:")
for f_ in sorted(os.listdir("tfjs_model")):
    size = os.path.getsize(os.path.join("tfjs_model", f_)) / 1024
    print(f"  tfjs_model/{f_}  ({size:,.0f} KB)")

In [ ]:
# ZIP으로 묶어서 다운로드
shutil.make_archive("tfjs_model", "zip", ".", "tfjs_model")
from google.colab import files
files.download("tfjs_model.zip")
print("tfjs_model.zip 다운로드 시작! (안 되면 왼쪽 📁 파일 탭에서 직접 다운로드)")

---
# 🎉 완료! 다음 단계

1. `tfjs_model.zip`의 압축을 풀어 **`tfjs_model` 폴더째** GitHub 저장소에 업로드
2. 같은 저장소에 웹앱 `custom_model_drive.html` 업로드
3. **Settings → Pages** 에서 GitHub Pages 활성화 (main / root)
4. 포니봇에서 `pony_ble_drive.py` 실행 → 웹앱 접속 → 블루투스 연결 → 카메라 시작!

자세한 순서는 실습 문서 **`lab_colab_transfer_learning.md`** 를 참고하세요.

> 🔁 **모델을 다시 학습하고 싶으면** 2단계부터 다시 실행하면 됩니다.
> 데이터를 보관하려면 `dataset` 폴더도 ZIP으로 내려받아 두세요:
> `shutil.make_archive("dataset", "zip", ".", "dataset")`